# Preambule
#### Goal
The goal of this session is to get familiar with the Bloomberg Python API.<br> 
This will be done by building a class containing a function which mimicks the behavior of the BDH Excel function.

#### What the function will do
Our BDH-like function should be able to : <br>
1 - Retrieve historical data <br>
2 - For as many tickers as possible <br>
3 - For as many fields as possible <br>
4 - Include the various options <br>
5 - And allow for the possibility to add overrides <br>

#### References 
https://data.bloomberglp.com/professional/sites/10/2017/03/BLPAPI-Core-Developer-Guide.pdf

# I. Dependencies
These are the libraries we will be using in this notebook. blpapi is the library used for Bloomberg data.

In [ ]:
import blpapi
import pandas as pd
import numpy as np
import datetime as dt

# II. Set up the Bloomberg names

We here create variables using the Name class within blpapi. <br> 
This will allow to write cleaner and more concise code when refering to strings with the api.<br>
Below are only the names required for our present work. Many more exist and you can refer to the different examples within the SDK for ones of interest to your task.

In [3]:
DATE = blpapi.Name("date")
ERROR_INFO = blpapi.Name("errorInfo")
EVENT_TIME = blpapi.Name("EVENT_TIME")
FIELD_DATA = blpapi.Name("fieldData")
FIELD_EXCEPTIONS = blpapi.Name("fieldExceptions")
FIELD_ID = blpapi.Name("fieldId")
SECURITY = blpapi.Name("security")
SECURITY_DATA = blpapi.Name("securityData")

# III. The BLP class
We now start to build our function within a dedicated class.<br>

A brief reminder on the class object in Python:<br>
- Classes must have a function called _\_init_\_() which is automatically executed at class initiation
- Classes can have one or several methods
- Class object need to be instaciated before using its methods

#### A. The init function

This function aims at starting the session and setting up the desired service 

#### B. The close session method:
Simply kills the session so no ghost connection remains. 

#### C. The BDP method:
3 steps: <br>
1- Create request<br>
2- Send request <br>
3- Extract data<br>




In [ ]:
class BLP():
   

   def __init__(self):
       
       self.session = blpapi.Session()

       if not self.session.start():
           print("Failed to start session.")
           return
       if not self.session.openService("//blp/refdata"):
           print("Failed to open //blp/refdata")
           return
       
       self.refDataSvc = self.session.getService('//blp/refdata')
       print('Session open')


   def bdh(self, strSecurity, strFields, startdate, enddate, per='DAILY', perAdj='CALENDAR',
           days='NON_TRADING_WEEKDAYS', fill='PREVIOUS_VALUE', curr=None):
       
       request = self.refDataSvc.createRequest('HistoricalDataRequest')
       
       if isinstance(strFields, str):
           strFields = [strFields]
       
       if isinstance(strSecurity, str):
           strSecurity = [strSecurity]
       
       for security in strSecurity:
           request.append('securities', security)
       
       for field in strFields:
           request.append('fields', field)
       
       request.set('startDate', startdate.strftime('%Y%m%d'))
       request.set('endDate', enddate.strftime('%Y%m%d'))
       request.set('periodicitySelection', per)
       request.set('periodicityAdjustment', perAdj)
       request.set('nonTradingDayFillOption', days)
       request.set('nonTradingDayFillMethod', fill)
       
       if curr:
           request.set('currency', curr)
       self.session.sendRequest(request)
       data = []
       keys = []
       
       while True:
           event = self.session.nextEvent()
           if event.eventType() not in [blpapi.Event.RESPONSE, blpapi.Event.PARTIAL_RESPONSE]:
               continue
           for msg in event:
               securityDataArray = msg.getElement('securityData')
               fieldData = securityDataArray.getElement('fieldData')
               fieldDataList = [fieldData.getValueAsElement(i) for i in range(fieldData.numValues())]
               df = pd.DataFrame()
               for fld in fieldDataList:
                   for v in [fld.getElement(i) for i in range(fld.numElements()) if fld.getElement(i).name() != 'date']:
                       df.loc[fld.getElementAsDatetime('date'), str(v.name())] = v.getValue()
               df.index = pd.to_datetime(df.index)
               df.replace('#N/A History', np.nan, inplace=True)
               keys.append(securityDataArray.getElementAsString('security'))
               data.append(df)
           if event.eventType() == blpapi.Event.RESPONSE:
               break
       
       if len(data) == 0:
           return pd.DataFrame()
       
       if isinstance(strSecurity, str) or len(strSecurity) == 1:
           data = pd.concat(data, axis=1)
           data.columns.name = 'Field'
       else:
           data = pd.concat(data, keys=keys, axis=1, names=['Security','Field'])
           data = data.swaplevel(axis=1)
           data = data.sort_index(axis=1, level=0)
           
       data.index.name = 'Date'
       return data
   
   def closeSession(self):
       self.session.stop()
       print("Session closed")

# IV. Tests

In [ ]:
blp = BLP()
strFields = ["PX_LAST","PX_VOLUME"]
tickers = ["GLE FP Equity", "BN FP Equity"]
startDate = dt.datetime(2020,10,1) 
endDate = dt.datetime(2020,11,3)
prices = blp.bdh(strSecurity=tickers, strFields = strFields, startdate = startDate, enddate = endDate)
BLP.closeSession()

Session open


In [47]:
prices

Field           PX_LAST                  PX_VOLUME              
Security   BN FP Equity GLE FP Equity BN FP Equity GLE FP Equity
Date                                                            
2020-10-01        55.26        11.048    1347922.0     5056724.0
2020-10-02        55.06        11.038    1136121.0     4921253.0
2020-10-05        55.26        11.388     800852.0     4262064.0
2020-10-06        55.14        12.152    1525322.0     9123734.0
2020-10-07        54.60        12.336    1274591.0     8280475.0
2020-10-08        55.00        12.594     997488.0     4990595.0
2020-10-09        55.54        12.416    1149669.0     4506467.0
2020-10-12        55.72        12.712     968110.0     6560510.0
2020-10-13        55.00        12.210    1823789.0     6225036.0
2020-10-14        55.50        12.322    1044651.0     4234361.0
2020-10-15        55.20        11.826    1754793.0     6236560.0
2020-10-16        53.30        12.058    3092582.0     6586723.0
2020-10-19        53.44        12.296     908428.0     2227175.0
2020-10-20        52.06        12.608    2317281.0     7325655.0
2020-10-21        50.60        12.392    2491266.0     5132329.0
2020-10-22        50.34        12.508    2584081.0     3928345.0
2020-10-23        51.12        12.750    1996675.0     5270262.0
2020-10-26        50.54        12.584    1311513.0     6023197.0
2020-10-27        49.11        11.944    2453662.0     6990562.0
2020-10-28        46.88        11.382    3596154.0    10900173.0
2020-10-29        46.83        11.350    2359932.0     6352473.0
2020-10-30        47.40        11.640    3026894.0     7073000.0
2020-11-02        47.80        12.124    2053931.0     6029462.0
2020-11-03        49.47        12.784    2486945.0     7722893.0